# 🌊 Open Oceanography – Interference of Crossing Seas

## 🎯 What You're Looking At

This notebook simulates a phenomenon known as **square seas** — a rare but dramatic pattern that forms when two wave trains intersect at right angles. Unlike the smooth, uniform swell often seen in open water, square seas appear as a grid-like pattern on the surface, caused by **interference** between wave systems traveling in perpendicular directions.

---

## 🌀 What This Notebook Does

We simulate two wave packets:

- A **red wave** traveling in the **negative X direction**
- A **blue wave** traveling in the **negative Y direction**

These waves are modeled using basic sinusoidal functions with Gaussian envelopes to give them localized shape.

We then:
- 🔵 Overlay the waves to create a **combined wave field**
- 🟢 Highlight regions where the waves **constructively interfere** (i.e., their amplitudes add up to a peak)
- 📹 Animate a 3D **flyover**, starting flat along the X axis, arcing over the center, and flattening out over the Y axis — and then reversing back

---

## 🌊 Why This Matters

Interacting wave systems are common in the ocean — especially near headlands, islands, or after storm systems pass. These crossings can lead to:

- Rougher seas and unpredictable boat handling
- Dangerous conditions for swimmers and surfers
    - I experienced this first hand paddling on this island.

The ocean is an endlessly complicated system, and I hope this illustrates just one of the many way the physics can be diffiuclt to understand and predict. 

By visualizing the **individual components** and their **combined result**, you’ll begin to see how **wave interference** explains many of the complex patterns observed on the real ocean surface that can have real world impacts. 

---

## 📚 Ocean Physics Concepts

- **Wave Interference**: When two wave trains meet, they can **add together (constructive)** or **cancel out (destructive)** depending on their phase.
- **Amplitude**: The vertical displacement of the wave from still water — here, you’ll see how amplitudes add spatially.
- **Wave Directionality**: The direction each wave travels affects how the overall pattern evolves in time.

---

This notebook is part of the *Open Oceanography* series, designed to help you see — and feel — the physics that shape our seas.
 
 As always feel free to adapt and play with the code to deeper your understanding!

In [40]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import imageio.v2 as imageio
import os

# Grid setup
x = np.linspace(0, 40, 200)
y = np.linspace(0, 40, 200)
X, Y = np.meshgrid(x, y)

# Wave parameters
wavelength = 5
k = 2 * np.pi / wavelength
omega = 2 * np.pi / 5
amplitude = 1

# Frame configuration
hold_frames = 15         # frames to pause on flat views (1.5s)
sweep_frames = 30        # frames to fly over origin
total_frames = 2 * hold_frames + 2 * sweep_frames

# Output folder
frame_dir = "square_sea_flyover_with_components"
os.makedirs(frame_dir, exist_ok=True)
filenames = []

for t in range(total_frames):
    time = t * 0.2
    wave_x = amplitude * np.exp(-(((X - 20)/10)**2 + ((Y - 20)/10)**2)) * np.cos(-k * X - omega * time)
    wave_y = amplitude * np.exp(-(((X - 20)/10)**2 + ((Y - 20)/10)**2)) * np.cos(-k * Y - omega * time)
    surface = wave_x + wave_y

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Plot individual wave components
    ax.plot_surface(X, Y, wave_x, cmap='Reds', edgecolor='none', alpha=0.4)
    ax.plot_surface(X, Y, wave_y, cmap='Blues', edgecolor='none', alpha=0.4)

    # Plot the combined wave
    ax.plot_surface(X, Y, surface, cmap='coolwarm', edgecolor='none', alpha=0.8)

    # Highlight constructive interference
    highlight = np.abs(surface) > 1.5
    ax.scatter(X[highlight], Y[highlight], surface[highlight], color='lime', s=1)

    # Axes labels and limits
    ax.set_zlim(-2, 2)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Amplitude')
    ax.set_title(f'Square Seas – Time = {time:.1f}s')
    # Direction arrows
    ax.quiver(40, 0, 2.5, -10, 0, 0, color="blue", arrow_length_ratio=0.2)
    ax.text(42, 0, 2.5, "Wave 1 ←", color="blue", fontsize=10)

    ax.quiver(0, 40, 2.5, 0, -10, 0, color="red", arrow_length_ratio=0.2)
    ax.text(0, 42, 2.5, "Wave 2 ↓", color="red", fontsize=10)
    # Camera flyover logic with pauses
    if t < hold_frames:
        azim = -180
        elev = 0
    elif t < hold_frames + sweep_frames:
        s = (t - hold_frames) / sweep_frames
        azim = -180 + 90 * s
        elev = 30 * np.sin(np.pi * s)
    elif t < hold_frames + sweep_frames + hold_frames:
        azim = -90
        elev = 0
    elif t < hold_frames + sweep_frames + hold_frames + sweep_frames:
        s = (t - hold_frames - sweep_frames - hold_frames) / sweep_frames
        azim = -90 + 90 * s
        elev = 30 * np.sin(np.pi * (1 - s))
    else:
        azim = -180
        elev = 0

    ax.view_init(elev=elev, azim=azim)

    # Save frame
    filename = os.path.join(frame_dir, f"frame_{t:03d}.png")
    plt.savefig(filename, bbox_inches='tight')
    plt.close()
    filenames.append(filename)

# Compile into GIF
gif_path = "square_sea_flyover_with_components.gif"
with imageio.get_writer(gif_path, mode='I', duration=0.1) as writer:
    for fname in filenames:
        writer.append_data(imageio.imread(fname))

print(f"GIF saved to {gif_path}")


GIF saved to square_sea_flyover_with_components.gif
